# B3 — RAG: image top-5, CLIP rerank (negative result)

**Pipeline:** `image → EVA-CLIP top-5 articles → pool all sections → EVA-CLIP text reranker picks top-3 → Qwen2.5-VL`.

Same retrieval as B4 — **only the reranker differs** (EVA-CLIP text encoder instead of a cross-encoder). Result: **0.243**, *below* even B1 (0.278) and no better than no-RAG (A = 0.245).

In [ ]:
import json, os, re

OUT = '../../outputs'
BASE = '/work/cvcs2026/encyclopedic'

for label, f in [('A', 'results.json'), ('B1', 'results_rag.json'),
                 ('B3 (CLIP)', 'results_B3.json'), ('B4 (cross)', 'results_B4.json')]:
    p = OUT + '/' + f
    if os.path.exists(p):
        print('%-12s overall %.4f' % (label, json.load(open(p))['accuracy_overall']))

## Same candidates as B4 — the reranker is the whole difference
B3 and B4 retrieve the exact same top-5 articles, so their retrieval recall is identical. Whatever gap exists between them is due only to which paragraphs the reranker selects.

In [ ]:
def norm(u):
    if not u:
        return u
    u = re.sub(r'^https?://', '', u.strip().lower())
    return u.replace('en.m.wikipedia.org', 'en.wikipedia.org').rstrip('/')

gt = {x['unique_id']: norm(x.get('wikipedia_url'))
      for x in json.load(open(BASE + '/encyclopedic_test_subset.json'))}
preds = [json.loads(l) for l in open(OUT + '/predictions_B3.jsonl') if l.strip()]

r5 = sum(1 for p in preds
         if gt.get(p['unique_id']) in
         [norm(c['wiki_url']) for c in (p.get('retrieved_context') or {}).get('candidates', [])])
print('B3 recall@5: %.1f%%  (identical to B4)' % (100 * r5 / len(preds)))
print('B3 accuracy 0.243  vs  B4 accuracy 0.360  -> reranker alone = +11.7 pts')

## Pros / cons / limitations

**Pros**
- Cheap: reuses the EVA-CLIP already loaded for image retrieval, no extra model.

**Cons**
- Actively *hurts*: the CLIP text encoder selects irrelevant paragraphs (e.g. photo-gallery links, unrelated species), so Qwen gets worse context than the plain article intro (B1).

**Why it fails**
- CLIP's text encoder is trained for image–caption matching, not question↔passage relevance, and truncates paragraphs to 77 tokens.

**Takeaway.** A bad reranker is worse than none. With the same retrieval, swapping CLIP for a cross-encoder (B4) turns this into the best strategy — reranking quality is the decisive factor.